In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import numpy as np
import pandas as pd
import os

Load Data

In [3]:
input_path = "D:/Projects with Git/AI-For-Health-IITGN-SRIP/Dataset"

patient_paths = [f.path for f in os.scandir(input_path) if f.is_file() and f.name.endswith('_dataset.npz')]

datasets = {}

for path in patient_paths:
    patient_id = os.path.basename(path).replace('_dataset.npz', '')

    with np.load(path, allow_pickle=True) as data:
        datasets[patient_id] = {
            "X": data["X"],
            "y_sleep": data["y_sleep"],
            "y_breath": data["y_breath"]
        }

Prepare categorical to numerical mapping

In [4]:
sleep_set = set()
breath_set = set()

for patient in datasets:
    sleep_set.update(np.unique(datasets[patient]["y_sleep"]))
    breath_set.update(np.unique(datasets[patient]["y_breath"]))

sleep_set = sorted(sleep_set)
breath_set = sorted(breath_set)

print("\nUnique Categories")
print(f"Sleep States:{sleep_set}")
print(f"Breathing Irregularities:{breath_set}")

sleep_map = {
    " A": 0,
    " N1": 1,
    " N2": 2,
    " N3": 3,
    " Movement": 4,
    " REM": 5,
    " Wake": 6
}

breath_map = {
    "Body event": 0,
    "Hypopnea": 1,
    "Mixed Apnea": 2,
    "Normal": 3,
    "Obstructive Apnea": 4
}

breath_map_inverse = {value:key for key, value in breath_map.items()}

print("\nError/Miss Check")
print(f"Sleep States: {set(sleep_set) - set(sleep_map.keys())}")
print(f"Breathing Irregularities: {set(breath_set) - set(breath_map.keys())}")


Unique Categories
Sleep States:[' A', ' Movement', ' N1', ' N2', ' N3', ' REM', ' Wake']
Breathing Irregularities:['Body event', 'Hypopnea', 'Mixed Apnea', 'Normal', 'Obstructive Apnea']

Error/Miss Check
Sleep States: set()
Breathing Irregularities: set()


Normalize the columns in X, apply mapping to y_breath and y_sleep.

\# Not used

\# Normalization should be applied during cross validation, only on the training set

```python
for key in datasets.keys():
    datasets[key]["X"][:, :, 0] = np.divide(
        np.subtract(datasets[key]["X"][:, :, 0], np.min(datasets[key]["X"][:, :, 0])), 
        np.max(datasets[key]["X"][:, :, 0]) - np.min(datasets[key]["X"][:, :, 0])
        )
    
    datasets[key]["X"][:, :, 1] = np.divide(
        np.subtract(datasets[key]["X"][:, :, 1], np.min(datasets[key]["X"][:, :, 1])), 
        np.max(datasets[key]["X"][:, :, 1]) - np.min(datasets[key]["X"][:, :, 1])
        )
    
    datasets[key]["X"][:, :, 2] = np.divide(
        np.subtract(datasets[key]["X"][:, :, 2], np.min(datasets[key]["X"][:, :, 2])), 
        np.max(datasets[key]["X"][:, :, 2]) - np.min(datasets[key]["X"][:, :, 2])
        )
    
    datasets[key]["y_breath"] = np.vectorize(lambda x: breath_map[x])(datasets[key]["y_breath"])
    datasets[key]["y_sleep"] = np.vectorize(lambda x: sleep_map[x])(datasets[key]["y_sleep"])
```



In [5]:
# Applying mappings to y columns
try:
    for key in datasets.keys():
        datasets[key]["y_breath"] = np.vectorize(lambda x: breath_map[x])(datasets[key]["y_breath"])
        datasets[key]["y_sleep"] = np.vectorize(lambda x: sleep_map[x])(datasets[key]["y_sleep"])
except KeyError as e:
    print(f"Mapping Error; Already Mapped: {e}")

Initialize DL framework

In [6]:
# constants
TIME_STEPS = 960
FEATURES = 3
NUM_SLEEP_STATES = len(sleep_map)
NUM_CLASSES = len(breath_map)

def build_model():

    # ----- SIGNAL INPUT BRANCH -----
    signal_input = layers.Input(shape=(TIME_STEPS, FEATURES), name="signal_input")

    x = layers.Conv1D(32, 3, activation='relu', padding='same')(signal_input)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling1D()(x)

    signal_features = x


    # ----- SLEEP INPUT BRANCH -----
    sleep_input = layers.Input(shape=(1,), name="sleep_input")

    s = layers.Embedding(NUM_SLEEP_STATES, 4)(sleep_input)
    s = layers.Flatten()(s)

    sleep_features = s


    # ----- MERGE BRANCHES -----
    merged = layers.Concatenate()([signal_features, sleep_features])


    # ----- CLASSIFIER -----
    x = layers.Dense(64, activation='relu')(merged)
    x = layers.Dropout(0.3)(x)

    output = layers.Dense(NUM_CLASSES, activation='softmax', name="breath_output")(x)


    # ----- MODEL OBJECT -----
    model = Model(
        inputs=[signal_input, sleep_input],
        outputs=output
    )


    # ----- COMPILE -----
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

model = build_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ signal_input        │ (None, 960, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 960, 32)   │        320 │ signal_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 480, 32)   │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 480, 64)   │      6,208 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 240, 64)   │          0 │ conv1d_1[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sleep_input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 240, 128)  │     24,704 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 4)      │         28 │ sleep_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv1d_2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 4)         │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 132)       │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,512 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ breath_output       │ (None, 5)         │        325 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 40,097 (156.63 KB)

 Trainable params: 40,097 (156.63 KB)

 Non-trainable params: 0 (0.00 B)

Prepare for Model Training

In [7]:
# Leave One Out Cross Validation
# from sklearn.model_selection import LeaveOneGroupOut
# logo = LeaveOneGroupOut()

all_y_breath_true = []
all_y_breath_pred = []

for patient_id in datasets.keys():
    X_test = datasets[patient_id]["X"]
    y_sleep_test = datasets[patient_id]["y_sleep"]
    y_breath_test = datasets[patient_id]["y_breath"]

    train_ids = [pid for pid in datasets.keys() if pid != patient_id]
    X_train = np.concatenate([datasets[pid]["X"] for pid in train_ids], axis=0)
    y_sleep_train = np.concatenate([datasets[pid]["y_sleep"] for pid in train_ids], axis=0)
    y_breath_train = np.concatenate([datasets[pid]["y_breath"] for pid in train_ids], axis=0)

    y_sleep_train = y_sleep_train.reshape(-1,1)
    y_sleep_test  = y_sleep_test.reshape(-1,1)

    # Apply class weights
    classes = np.unique(y_breath_train)          # [0,1,2,3,4]
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_breath_train
    )
    class_weight_dict = dict(zip(classes, class_weights))

    # Normalization
    scaler_0 = StandardScaler()
    scaler_1 = StandardScaler()
    scaler_2 = StandardScaler()

    X_train[:,:,0] = scaler_0.fit_transform(X_train[:,:,0])
    X_train[:,:,1] = scaler_1.fit_transform(X_train[:,:,1])
    X_train[:,:,2] = scaler_2.fit_transform(X_train[:,:,2])

    X_test[:,:,0] = scaler_0.transform(X_test[:,:,0])
    X_test[:,:,1] = scaler_1.transform(X_test[:,:,1])
    X_test[:,:,2] = scaler_2.transform(X_test[:,:,2])

    print(f"\nTraining on {len(train_ids)} patients, Testing on {patient_id}")

    model = build_model()
    model.fit(
        [X_train, y_sleep_train],
        y_breath_train,
        epochs=25,
        class_weight=class_weight_dict,
        validation_split=0.2
    )

    y_pred = model.predict([X_test, y_sleep_test])
    y_pred_classes = np.argmax(y_pred, axis=1)

    report = classification_report(y_breath_test, y_pred_classes)
    
    all_y_breath_true.extend(y_breath_test)
    all_y_breath_pred.extend(y_pred_classes)



Training on 4 patients, Testing on AP01
Epoch 1/25
175/175 ━━━━━━━━━━━━━━━━━━━━ 17s 65ms/step - accuracy: 0.6886 - loss: 1.8763 - val_accuracy: 0.7693 - val_loss: 1.0111
Epoch 2/25
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 64ms/step - accuracy: 0.7164 - loss: 2.3134 - val_accuracy: 0.5817 - val_loss: 1.1256
Epoch 3/25
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.6541 - loss: 1.5688 - val_accuracy: 0.4900 - val_loss: 1.2459
Epoch 4/25
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 0.6838 - loss: 1.3288 - val_accuracy: 0.2264 - val_loss: 1.3383
Epoch 5/25
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - accuracy: 0.4747 - loss: 1.1016 - val_accuracy: 0.3410 - val_loss: 1.2448
Epoch 6/25
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - accuracy: 0.4742 - loss: 1.2212 - val_accuracy: 0.5494 - val_loss: 1.1484
Epoch 7/25
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - accuracy: 0.3681 - loss: 1.0131 - val_accuracy: 0.2794 - val_loss: 1.2965
Epoch 8/25
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step 

c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib


Training on 4 patients, Testing on AP02
Epoch 1/25
176/176 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.8426 - loss: 2.0930 - val_accuracy: 0.7704 - val_loss: 1.2042
Epoch 2/25
176/176 ━━━━━━━━━━━━━━━━━━━━ 9s 48ms/step - accuracy: 0.9321 - loss: 1.9124 - val_accuracy: 0.7704 - val_loss: 1.0075
Epoch 3/25
176/176 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - accuracy: 0.8288 - loss: 1.8415 - val_accuracy: 0.5103 - val_loss: 1.2732
Epoch 4/25
176/176 ━━━━━━━━━━━━━━━━━━━━ 8s 46ms/step - accuracy: 0.7264 - loss: 1.2936 - val_accuracy: 0.2786 - val_loss: 1.2560
Epoch 5/25
176/176 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step - accuracy: 0.6088 - loss: 1.4037 - val_accuracy: 0.1990 - val_loss: 1.2761
Epoch 6/25
176/176 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.4902 - loss: 1.1717 - val_accuracy: 0.2758 - val_loss: 1.1718
Epoch 7/25
176/176 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - accuracy: 0.3889 - loss: 1.1907 - val_accuracy: 0.1485 - val_loss: 1.3605
Epoch 8/25
176/176 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - a

c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib


Training on 4 patients, Testing on AP03
Epoch 1/25
178/178 ━━━━━━━━━━━━━━━━━━━━ 10s 40ms/step - accuracy: 0.7350 - loss: 2.1870 - val_accuracy: 0.7720 - val_loss: 1.2468
Epoch 2/25
178/178 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - accuracy: 0.6965 - loss: 1.6845 - val_accuracy: 0.3681 - val_loss: 1.3145
Epoch 3/25
178/178 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - accuracy: 0.7137 - loss: 1.4643 - val_accuracy: 0.2379 - val_loss: 1.4169
Epoch 4/25
178/178 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.6847 - loss: 1.2621 - val_accuracy: 0.3075 - val_loss: 1.2658
Epoch 5/25
178/178 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.6688 - loss: 1.1818 - val_accuracy: 0.1633 - val_loss: 1.3777
Epoch 6/25
178/178 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.5330 - loss: 1.0337 - val_accuracy: 0.2632 - val_loss: 1.2810
Epoch 7/25
178/178 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - accuracy: 0.5443 - loss: 1.2156 - val_accuracy: 0.1527 - val_loss: 1.3513
Epoch 8/25
178/178 ━━━━━━━━━━━━━━━━━━━━ 8s 45ms/step - 

c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib


Training on 4 patients, Testing on AP04
Epoch 1/25
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 47ms/step - accuracy: 0.7088 - loss: 1.6179 - val_accuracy: 0.7686 - val_loss: 1.3755
Epoch 2/25
172/172 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.7526 - loss: 1.4398 - val_accuracy: 0.0197 - val_loss: 1.5627
Epoch 3/25
172/172 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.7235 - loss: 1.0466 - val_accuracy: 0.7686 - val_loss: 1.2784
Epoch 4/25
172/172 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - accuracy: 0.7344 - loss: 1.6273 - val_accuracy: 0.7686 - val_loss: 0.7400
Epoch 5/25
172/172 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - accuracy: 0.7170 - loss: 1.6404 - val_accuracy: 0.7686 - val_loss: 1.3849
Epoch 6/25
172/172 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.7033 - loss: 1.8356 - val_accuracy: 0.0189 - val_loss: 1.9245
Epoch 7/25
172/172 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.7048 - loss: 1.1729 - val_accuracy: 0.6121 - val_loss: 1.2949
Epoch 8/25
172/172 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - 

c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib


Training on 4 patients, Testing on AP05
Epoch 1/25
181/181 ━━━━━━━━━━━━━━━━━━━━ 10s 41ms/step - accuracy: 0.5503 - loss: 5.2172 - val_accuracy: 0.8850 - val_loss: 0.3973
Epoch 2/25
181/181 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step - accuracy: 0.5936 - loss: 3.3601 - val_accuracy: 0.7805 - val_loss: 1.0011
Epoch 3/25
181/181 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - accuracy: 0.5926 - loss: 1.4441 - val_accuracy: 0.8850 - val_loss: 0.4189
Epoch 4/25
181/181 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - accuracy: 0.5796 - loss: 3.6951 - val_accuracy: 0.8850 - val_loss: 0.5355
Epoch 5/25
181/181 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step - accuracy: 0.6397 - loss: 1.7217 - val_accuracy: 0.8850 - val_loss: 0.5542
Epoch 6/25
181/181 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.6731 - loss: 1.5746 - val_accuracy: 0.8843 - val_loss: 0.6127
Epoch 7/25
181/181 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - accuracy: 0.6635 - loss: 1.1123 - val_accuracy: 0.8366 - val_loss: 0.6589
Epoch 8/25
181/181 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - 

c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Displaying Results

In [9]:
# Calculate weighted average classification report

all_y_breath_true = np.array(all_y_breath_true)
all_y_breath_pred = np.array(all_y_breath_pred)

final_report_dict = classification_report(all_y_breath_true, 
                                          all_y_breath_pred, 
                                          digits=4, 
                                          output_dict=True)

mapped_report = {}
for label, metrics in final_report_dict.items():
    if label.isdigit():
        mapped_report[breath_map_inverse[int(label)]] = metrics
    else:
        mapped_report[label] = metrics  # macro avg, weighted avg, accuracy

# Put in DataFrame with desired row order
row_order = list(breath_map.keys()) + ["macro avg", "weighted avg", "accuracy"]
df_report = pd.DataFrame(mapped_report).T.reindex(row_order)
# ensure support is int for print formatting
if "support" in df_report.columns:
    df_report["support"] = df_report["support"].astype(int)

# Build printed rows manually for perfect control
print("               precision    recall  f1-score   support")
for idx in row_order:
    if idx == "macro avg":
        # blank line before macro avg
        print()
    row = df_report.loc[idx]
    if idx == "accuracy":
        # accuracy row prints with placeholder fields for recall/f1
        print(f"{idx:<16s} {row['precision']:.4f}      -     -   {int(row['support'])}")
    else:
        print(f"{idx:<16s} {row['precision']:.4f}    {row['recall']:.4f}    {row['f1-score']:.4f}   {int(row['support'])}")



               precision    recall  f1-score   support
Body event       0.0006    0.6667    0.0013   3
Hypopnea         0.0919    0.2652    0.1365   592
Mixed Apnea      0.0000    0.0000    0.0000   2
Normal           0.9408    0.1602    0.2738   8039
Obstructive Apnea 0.0008    0.0061    0.0014   164

macro avg        0.2068    0.2196    0.0826   8800
weighted avg     0.8657    0.1645    0.2593   8800
accuracy         0.1645      -     -   0
